In [1]:
%%javascript
IPython.OutputArea.prototype._should_scroll = function(lines) {
    return false; // disable scroll bar when displaying Folium map
}

<IPython.core.display.Javascript object>

# Assignment 2

Before working on this assignment please read these instructions fully. In the submission area, you will notice that you can click the link to **Preview the Grading** for each step of the assignment. This is the criteria that will be used for peer grading. Please familiarize yourself with the criteria before beginning the assignment.

The data for this assignment comes from a subset of The National Centers for Environmental Information (NCEI) [Global Historical Climatology Network daily (GHCNd)](https://www.ncei.noaa.gov/products/land-based-station/global-historical-climatology-network-daily) (GHCN-Daily). The GHCN-Daily is comprised of daily climate records from thousands of land surface stations across the globe - it's a wonderfully large dataset to play with! In particular, you will be asked to use data from the Ann Arbor Michigan location (my home!). and this is stored in the file: `assets/fb441e62df2d58994928907a91895ec62c2c42e6cd075c2700843b89.csv`

Each row in this datafile corresponds to a single observation from a weather station, and has the following variables:
* **id** : station identification code
* **date** : date in YYYY-MM-DD format (e.g. 2012-01-24 = January 24, 2012)
* **element** : indicator of element type
    * TMAX : Maximum temperature (tenths of degrees C)
    * TMIN : Minimum temperature (tenths of degrees C)
* **value** : data value for element (tenths of degrees C)

For this assignment, you must:

1. Read the documentation and familiarize yourself with the dataset, then write a python notebook which plots line graphs of the record high and record low temperatures by day of the year over the period 2005-2014. The area between the record high and record low temperatures for each day should be shaded.
2. Overlay a scatter of the 2015 data for any points (highs and lows) for which the ten year record (2005-2014) record high or record low was broken in 2015. (Based on the graph, do you think extreme weather is getting more frequent in 2015?)
3. Watch out for leap days (i.e. February 29th), it is reasonable to remove these points from the dataset for the purpose of this visualization.
4. Make the visual nice! Leverage principles from the first module in this course when developing your solution. Consider issues such as legends, labels, and chart junk.

I've written some steps I think would be good to go through, but there are other ways to solve this assignment so feel free to explore the pandas library! What I really want to see is an image that looks like this sketch I drew at my desk:

![](assets/chris_sketch.png)

In [126]:
#  I'll be using the folium package to render the data into a map in Jupyter.

import folium
import pandas as pd

# get the location information for this dataset
df = pd.read_csv('assets/BinSize_d400.csv')
station_locations_by_hash = df[df['hash'] == 'fb441e62df2d58994928907a91895ec62c2c42e6cd075c2700843b89']

# get longitude and lattitude to plot
lons = station_locations_by_hash['LONGITUDE'].tolist()
lats = station_locations_by_hash['LATITUDE'].tolist()

# plot on a beautiful folium map
my_map = folium.Map(location = [lats[0], lons[0]], height = 500,  zoom_start = 9)
for lat, lon in zip(lats, lons):
    folium.Marker([lat, lon]).add_to(my_map)

# render map in Jupyter
display(my_map)

## Step 1
Load the dataset and transform the data into Celsius (refer to documentation) then extract all of the rows which have minimum or maximum temperatures.

__hint: when I did this step I had two DataFrame objects, each with ~80,000 entries in it__

In [127]:
import pandas as pd
df = pd.read_csv('assets/fb441e62df2d58994928907a91895ec62c2c42e6cd075c2700843b89.csv')
df.head()

,ID,Date,Element,Data_Value
0,USW00094889,2014-11-12,TMAX,22
1,USC00208972,2009-04-29,TMIN,56
2,USC00200032,2008-05-26,TMAX,278
3,USC00205563,2005-11-11,TMAX,139
4,USC00200230,2014-02-27,TMAX,-106


In [128]:
df['Date'] = pd.to_datetime(df['Date'], format='%Y-%m-%d')
print(df.dtypes)
df = df.sort_values(by='Date')
df.head()

ID                    object
Date          datetime64[ns]
Element               object
Data_Value             int64
dtype: object


,ID,Date,Element,Data_Value
60995,USW00004848,2005-01-01,TMIN,0
17153,USC00207320,2005-01-01,TMAX,150
17155,USC00207320,2005-01-01,TMIN,-11
10079,USW00014833,2005-01-01,TMIN,-44
10073,USW00014833,2005-01-01,TMAX,33


In [129]:
# In this code cell, transform the Data_Value column
df['Data_Value'] = df['Data_Value'] / 10
df_high = df[df.Element == 'TMAX']
df_low = df[df.Element == 'TMIN']
print(df_high.shape, df_low.shape)
df_high.head()

(83063, 4) (82022, 4)


,ID,Date,Element,Data_Value
17153,USC00207320,2005-01-01,TMAX,15.0
10073,USW00014833,2005-01-01,TMAX,3.3
60994,USW00004848,2005-01-01,TMAX,13.3
39454,USC00205563,2005-01-01,TMAX,2.8
18049,USW00014853,2005-01-01,TMAX,5.6


## Step 2
In order to visualize the data we would plot the min and max data for each day of the year between the years 2005 and 2014 across all weather stations. But we also need to find out when the min or max temperature in 2015 falls below the min or rises above the max for the previous decade.

If you did step 1 you have two Series objects with min and max times for the years 2005 through 2015. You can use Pandas `groupby` to create max and min temperature Series objects across all weather stations for each day of these years, and you can deal with the records for February 29 (the leap year) by dropping them.

__hint: when I finished this step, I had two DataFrame objects, each with exactly 4015 observations in them__

In [130]:
# create a DataFrame of maximum temperature by date
df_high_agg = df_high.groupby('Date').agg({'Data_Value': ['max']}).reset_index()
df_high_agg.columns = ['Date', 'max']
#print(df_high_agg.loc[((df_high_agg.Date.dt.month == 2) & (df_high_agg.Date.dt.day == 29))])
df_high_agg = df_high_agg.loc[~((df_high_agg.Date.dt.month == 2) & (df_high_agg.Date.dt.day == 29))]
print(df_high_agg.shape)
df_high_agg.head()
# create a DataFrame of minimum temperatures by date
df_low_agg = df_low.groupby('Date').agg({'Data_Value': ['min']}).reset_index()
df_low_agg.columns = ['Date', 'min']
#print(df_low_agg.loc[((df_low_agg.Date.dt.month == 2) & (df_low_agg.Date.dt.day == 29))])
df_low_agg = df_low_agg.loc[~((df_low_agg.Date.dt.month == 2) & (df_low_agg.Date.dt.day == 29))]
print(df_low_agg.shape)
df_low_agg.head()

(4015, 2)
(4015, 2)


,Date,min
0,2005-01-01,-5.6
1,2005-01-02,-5.6
2,2005-01-03,0.0
3,2005-01-04,-3.9
4,2005-01-05,-9.4


## Step 3
Now that you have grouped the daily max and min temperatures for each day of the years 2005 through 2015, you can separate out the data for 2015. Then you can use the Pandas `groupby` function to find the max and min of the temperature data for each __day of the year__ for the 2005-2014 data.

__hint: at the end of this step I had two DataFrames, one of maximum and the other of minimum values, which each had 365 observations in them. I also had another pair of similar DataFrames but only for the year 2015.__

In [131]:
def process_df(df, agg_func):
    df_agg = df.groupby('Date').agg({'Data_Value': [agg_func]}).reset_index()
    df_agg.columns = ['Date', 'Temp']
    df_agg = df_agg.loc[~((df_agg.Date.dt.month == 2) & (df_agg.Date.dt.day == 29))]
    return df_agg

def process_df1(df, agg_func):
    df_agg = process_df(df, agg_func)
    #print(df_agg.head(10))
    df_agg = df_agg[['Temp']]
    df_agg['Date'] = pd.date_range(start='1/1/2015', end='31/12/2015')
    df_agg = df_agg[['Date', 'Temp']]
    print(df_agg.shape)
    return df_agg    

def process_df2(df, agg_func):
    df = process_df(df, agg_func)
    df['month'], df['day'] = df.Date.dt.month, df.Date.dt.day
    df_agg = df.groupby(['month', 'day']).agg({'Temp': [agg_func]}).reset_index()
    df_agg.columns = ['month', 'day', 'Temp']
    df_agg = df_agg[['Temp']]
    df_agg['Date'] = pd.date_range(start='1/1/2015', end='31/12/2015')
    df_agg = df_agg[['Date', 'Temp']]
    print(df_agg.shape)
    return df_agg

# calculate the minimum and maximum values for the day of the year for 2005 through 2014
df_high_05_14 = df_high.loc[df_high.Date.dt.year != 2015]
df_high_05_14_agg = process_df2(df_high_05_14, 'max')
df_high_05_14_agg.head()
df_low_05_14 = df_low.loc[df_low.Date.dt.year != 2015]
df_low_05_14_agg = process_df2(df_low_05_14, 'min')
df_low_05_14_agg.head()

# calculate the minimum and maximum values for the day of the year for 2015
df_high_15 = df_high.loc[df_high.Date.dt.year == 2015]
df_high_15_agg = process_df1(df_high_15, 'max')
df_high_15_agg.head()
df_low_15 = df_low.loc[df_low.Date.dt.year == 2015]
df_low_15_agg = process_df1(df_low_15, 'min')
df_low_15_agg.head()

(365, 2)
(365, 2)
(365, 2)
(365, 2)


,Date,Temp
0,2015-01-01,-13.3
1,2015-01-02,-12.2
2,2015-01-03,-6.7
3,2015-01-04,-8.8
4,2015-01-05,-15.5


## Step 4
Now it's time to plot! You need to explore matplotlib in order to plot line graphs of the min and max temperatures for the years 2005 through 2014 and to scatter plot __only__ the daily 2015 temperatures that exceeded those values.

In [ ]:
import matplotlib.pyplot as plt
from calendar import month_abbr

# put your plotting code here!

#font = {'family' : 'sans',
#        'weight' : 'bold',
#        'size'   : 22}

import seaborn as sns
#matplotlib.rc('font', **font)

plt.rcParams.update({'font.size': 20,})

# Update the index to be the desired display format for x-axis
#df_low_05_14_agg.index = df_low_05_14_agg.apply(lambda x: month_abbr[x[0].month], axis=1)
#df_high_05_14_agg.index = [month_abbr[i] for i in df_high_05_14_agg.Date.dt.month.values]

#df_low_05_14_agg.plot(x='Date', y='Temp', kind='line')
#df_high_05_14_agg.plot(x='Date', y='Temp', kind='line')

fig, ax = plt.subplots(figsize=(15,7))

m = sns.lineplot(x='Date', y='Temp', data=df_low_05_14_agg, label = 'min temp 2005-2014', ax=ax)
M = sns.lineplot(x='Date', y='Temp', data=df_high_05_14_agg, label = 'max temp 2005-2014', ax=ax) #, color='r')

line = M.get_lines()

xdata, ydata_m, ydata_M = line[0].get_xdata(), line[0].get_ydata(), line[1].get_ydata()
ax.fill_between(xdata, ydata_m, ydata_M, color='gray', alpha=.25)
plt.grid()

mdata = np.array([month_abbr[i] for i in df_high_05_14_agg.Date.dt.month.values])
indices = np.array([0, 31, 59, 90, 120, 151, 181, 212, 243, 273, 304, 334])

ax.set_xticks(xdata[indices], mdata[indices], rotation=90)  # Set text labels and properties.
ax.set_title('Daily Temperature (average) at Ann Arbor Michigan 2005-2014 vs. 2015', size=25)

df_extreme_low = pd.merge(df_low_05_14_agg, df_low_15_agg, on='Date')
df_extreme_low = df_extreme_low[df_extreme_low.Temp_x > df_extreme_low.Temp_y]
sns.scatterplot(data=df_extreme_low, x=xdata[df_extreme_low.index], y='Temp_y', s=100, facecolor='skyblue', edgecolor='k', label='min temp 2015')

df_extreme_high = pd.merge(df_high_05_14_agg, df_high_15_agg, on='Date')
df_extreme_high = df_extreme_high[df_extreme_high.Temp_x < df_extreme_high.Temp_y]
#print(df_extreme_high)
sns.scatterplot(data=df_extreme_high, x=xdata[df_extreme_high.index], y='Temp_y', s=100, facecolor='red', edgecolor='k', label='max temp 2015')

#sns.scatterplot(x =df_sat_test['t'], y = np.array(test_ppc.observed_data.obs), label = 'True Value')
leg = plt.legend()
leg.get_frame().set_edgecolor('k')
leg.get_frame().set_linewidth(1)

plt.xlabel('Month')
plt.ylabel('Temperature')
plt.show()

#plt.fill_between(range(len(df_low_05_14_agg)), df_low_05_14_agg, df_high_05_14_agg, color='C0', alpha=0.2)